# JSON preparation demo

Prepare condition-classification training and holdout JSONL from the bundled positive and negative records. The notebook calls the main dataset preparation module through `run_demo.py`. No API key or training job is needed.

Install from the repository root with `python -m pip install -e ".[curation,datasets,notebook]"`.

## Inputs and setup

Run the cells in order. The following paths are relative to `Demo/02_json_preparation/`; all required files are included.

| Input | Location | Contents |
| --- | --- | --- |
| Cleaned positives, CSV | `input/positive_cleaned.csv` | The expected stage 6 output from the cleaning demo. |
| Cleaned negatives, CSV | `input/negative_cleaned.csv` | Enumerated and curated negative-condition records. |
| Publication years, CSV | `input/publication_years.csv` | `DOI` and `Publication Year` columns. |
| Holdout conditions, JSON | `input/forced_questions.json` | Condition definitions selected by `config.json`. |
| Classification prompt, TXT | `../../prompts/dataset_classification.txt` | The main workflow's classification prompt. |

To use the output of the cleaning demo, pass `positive_csv=REPO / "Demo/01_data_cleaning/outputs/mof_extraction_1_2_3_4_5_6.csv"` to `run_demo` below. To test different cleaned tables, select their paths in `config.json` and use `check=False`; `check=True` compares with the supplied dataset only. Keep the input column schemas and provide matching DOI/year metadata.

Outputs are saved together under `outputs/`: `mof_ft_train.jsonl`, `mof_ft_holdout.jsonl`, `mof_ft_split_assignments.csv`, and summary files. The split is computed locally; it does not start a training job.

Implementation: [demo runner](run_demo.py) and [dataset preparation](../../src/mofinder/datasets/prepare.py). See the [source-to-code guide](../../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import runpy
import pandas as pd

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "mofinder").is_dir():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from within the MOFinder repository.")

import json
DEMO = REPO / "Demo" / "02_json_preparation"


## 1. Inspect the inputs

In [ ]:
positive = pd.read_csv(DEMO / "input" / "positive_cleaned.csv")
negative = pd.read_csv(DEMO / "input" / "negative_cleaned.csv")
pd.DataFrame({"label": ["P", "N"], "input_rows": [len(positive), len(negative)]})


## 2. Prepare and verify the datasets

The seed-42 split groups records by metal precursor, linker set, and solvent set. The bundled positive input is the expected output of the cleaning demo.

In [ ]:
run_demo = runpy.run_path(str(DEMO / "run_demo.py"))["run"]
summary = run_demo(check=True)
pd.DataFrame({name: summary["labels"][name] for name in ["train", "holdout"]})


## 3. Inspect a training record and its split assignment

In [ ]:
with (DEMO / "outputs" / "mof_ft_train.jsonl").open(encoding="utf-8") as stream:
    example = json.loads(next(stream))
example


In [ ]:
assignments = pd.read_csv(DEMO / "outputs" / "mof_ft_split_assignments.csv")
assignments[["source_row_id", "doi_norm", "publication_year", "is_success", "split"]].head(12)


The full split summary records input hashes, filtering counts, and preparation settings. The small demo datasets are separate from the full research training and validation files.